# PitWall-X: Tyre Degradation, Fuel-Corrected Pace & Pit Strategy

Pipeline: Setup -> Data Loading -> EDA -> Preprocessing & Feature Engineering -> Generalising -> Modelling -> Evaluation -> Decision (MDP framing).

Developed first on a single race (Bahrain 2023), then generalised across multiple races within the same regulation era (2022-2025) via a reusable function.

## Phase 0: Setup

In [4]:
import os
import json
import time
import warnings
import datetime as dt
import logging
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

fastf1.Cache.enable_cache("../cache/")

## Phase 1: Data Loading — Bahrain Grand Prix 2023

We are starting by analyzing one race, preferably in a race where the circuit is pretty stable with overtaking opportunities, balanced pace etc. One of the choices is the Bahrain circuit, it's special because it's where tyre degradation appears to be and there are lots of overtakings, one of the reasons why it's the pre-season testing circuit so it's where teams actually test their cars before the season begins.

We load the race data (laps and results).

In [ ]:
bahrain2023_race = fastf1.get_session(2024, "Netherlands", "R")
bahrain2023_race.load()
laps = bahrain2023_race.laps
results = bahrain2023_race.results

## Phase 2: Exploratory Data Analysis (EDA)

### 2.1 Inspecting the raw data
We inspect the shape, columns, data types, and a sample of rows to understand what FastF1 gives us.

In [ ]:
print(laps.shape)
print(laps.columns)
print(laps.dtypes)
print(laps.head())

We also inspect the results table, which holds the per-driver finishing classification (the Status column) used to identify finishers vs retirees.

In [ ]:
print(results.shape)
print(results.columns)
print(results["Status"].unique())

### 2.2 Cleaning, missing values, and stint overview
Before going forward we have to clean our data, there are laps that don't count, either in/out laps, laps under safety car, invalid laps etc. We convert lap times to seconds, cast the stint to an integer, keep only accurate laps, report missing values, and print each driver's stint lengths.

In [ ]:
laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()
laps["Stint"] = laps["Stint"].astype(int)
clean_laps = laps[laps["IsAccurate"] == True].copy()
drivers = clean_laps["Driver"].unique()
print(f" Missed values: {clean_laps.isna().sum()}")
for d in drivers:
    ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
    print("Driver: ", d)
    print(ds)
    for s in ds:
        print(f"stint {s}: ", clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)].shape[0])

### 2.3 Identifying finishers (official classification)
We use the official Status from the results table to keep finishers (Finished + Lapped) and drop retirees, rather than a lap-count heuristic which would confuse lapped finishers with retirements.

In [ ]:
finisher_abbrevs = results[results["Status"].isin(["Finished", "Lapped"])]["Abbreviation"].tolist()
finisher_laps = clean_laps[clean_laps["Driver"].isin(finisher_abbrevs)].copy()
print(f"All drivers: {clean_laps['Driver'].nunique()}, finishers: {finisher_laps['Driver'].nunique()}")
drivers = finisher_laps["Driver"].unique()
print(f"Finishers: {set(drivers)}")

print(results["Status"].unique())

### 2.4 Correlation check
We examine the correlation between TyreLife and FuelMass, pooled across all laps vs. within a single stint, to demonstrate the multicollinearity and how pit-stop resets break it. (FuelMass is computed here so the correlation can be examined; the fuel model is detailed in Phase 3.)

In [ ]:
START_FUEL = 109.0
FUEL_BURN = START_FUEL/57
clean_laps["FuelMass"] = START_FUEL - (FUEL_BURN * clean_laps["LapNumber"]) + 1
finisher_laps["FuelMass"] = START_FUEL - (FUEL_BURN * finisher_laps["LapNumber"]) + 1

print("Overall correlation:")
print(clean_laps[["TyreLife", "FuelMass", "LapTimeSeconds"]].corr(), "\n")

print("Overall correlation (Finished drivers only):")
print(finisher_laps[["TyreLife", "FuelMass", "LapTimeSeconds"]].corr(), "\n")


ver_stint2 = clean_laps[(clean_laps["Driver"] == "VER") & (clean_laps["Stint"] == 2)]
print("Verstappen Stint 2 correlation:")
print(ver_stint2[["TyreLife", "FuelMass", "LapTimeSeconds"]].corr())

Within a stint, TyreLife and FuelMass are perfectly collinear (r = -1.00), so fuel and degradation cannot be separated from a single stint. Pooled across the race the pit-stop resets reduce the correlation (~-0.53), which is what allows the multi-stint regression to identify the two coefficients separately. The near-zero raw correlation between tyre age and lap time is the statistical fingerprint of fuel masking degradation, motivating the fuel correction. We also verified the pooled correlation is robust to removing retirees (negligible change), confirming it reflects genuine race structure rather than contamination.

### 2.5 Distribution and outliers

In [ ]:
plt.xlabel("Lap time (s)")
plt.ylabel("Frequency")
plt.title("Distribution of lap times in Bahrain 2023 (All drivers)")
plt.hist(clean_laps["LapTimeSeconds"], bins=40)
plt.show()

In [ ]:
plt.xlabel("Lap time (s)")
plt.ylabel("Frequency")
plt.title("Distribution of lap times in Bahrain 2023 (Finishers only)")
plt.hist(finisher_laps["LapTimeSeconds"], bins=40)
plt.show()

The distribution is right-skewed: a hard floor near optimal pace, a long tail of slower laps from traffic and errors (fast is bounded, slow is not). It also shows mild bimodality (a secondary cluster ~101s), likely reflecting distinct pace regimes (heavy-fuel early laps and/or harder compounds), consistent with our finding that fuel and compound strongly shape lap time. The bimodality is more evidence that raw lap time is a mixture of regimes (fuel, compound, phase) rather than a clean degradation signal.

### 2.6 Raw degradation plots per stint
Plotting lap time against tyre age for every driver-stint (before any fuel correction) to visualise the raw degradation behaviour.

In [ ]:
for d in drivers:
    ds = finisher_laps[finisher_laps["Driver"] == d].Stint.unique()
    for s in ds:
        driver_stint = finisher_laps[(finisher_laps["Driver"] == d) & (finisher_laps["Stint"] == s)]
        compound = driver_stint["Compound"].iloc[0]
        plt.xlabel("Tyre Age")
        plt.ylabel("Lap Time")
        plt.title(f"{d} Stint {s} Compound: {compound} Bahrain 2023 laptime evolution by tyre age (Lower is faster)")
        plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
        plt.scatter(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
        plt.savefig(f"../figs/bahrain2023_{d}_stint{s}_{compound}_tyredeg_evo_scatter.png")
        plt.cla()
plt.close()

## Phase 3: Preprocessing & Feature Engineering

### 3.1 Estimating the fuel coefficient (gamma)
We regress lap time on tyre age and fuel mass per driver (finishers only), and average the fuel coefficient across drivers. The pit-stop resets decouple tyre age from fuel mass, making the two effects identifiable.

In [ ]:
sec_per_kg_fuel = []
for d in drivers:
    driver_laps = finisher_laps[finisher_laps["Driver"] == d].copy()
    y = driver_laps['LapTimeSeconds']
    driver_laps["FuelMass"] = START_FUEL - (FUEL_BURN * driver_laps["LapNumber"]) + 1
    X = driver_laps[["TyreLife", "FuelMass"]]
    laptime_predictor = LinearRegression()
    laptime_predictor.fit(X, y)
    print(f"Driver: {d}, deg_rate = {laptime_predictor.coef_[0]}, fuel_coef ={laptime_predictor.coef_[1]}")
    sec_per_kg_fuel.append(laptime_predictor.coef_[1])
gamma = np.mean(sec_per_kg_fuel)
per_lap_gain = gamma * FUEL_BURN
race_cum_gain = per_lap_gain * 57
print(f"Mean of seconds earned per kg: {gamma}")
print(f"Per-lap gain in seconds: {per_lap_gain}")
print(f"Race gain in seconds: {race_cum_gain}")

### 3.2 Applying the fuel correction
We compute fuel mass on every clean lap and subtract the fuel time penalty to obtain the fuel-corrected lap time, then re-plot degradation per stint on the corrected times.

In [ ]:
finisher_laps["FuelCorrectedLapTime"] = finisher_laps["LapTimeSeconds"] - gamma * finisher_laps["FuelMass"]
print(finisher_laps[["LapNumber", "Driver", "LapTimeSeconds", "FuelMass", "FuelCorrectedLapTime"]])
for d in drivers:
    ds = finisher_laps[finisher_laps["Driver"] == d].Stint.unique()
    for s in ds:
        driver_stint = finisher_laps[(finisher_laps["Driver"] == d) & (finisher_laps["Stint"] == s)]
        compound = driver_stint["Compound"].iloc[0]
        plt.xlabel("Tyre Age")
        plt.ylabel("Lap Time")
        plt.title(f"{d} Stint {s} Compound: {compound} Bahrain 2023 laptime evolution by tyre age (Fuel-corrected) (Lower is faster)")
        plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['FuelCorrectedLapTime']))
        plt.scatter(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['FuelCorrectedLapTime']))
        plt.savefig(f"../figs/bahrain2023_{d}_stint{s}_{compound}_tyredeg_evo_fuel_corrected_scatter.png")
        plt.cla()
plt.close()

### 3.3 Compound encoding (apply once)
Compound is a categorical feature (SOFT/MEDIUM/HARD) with no guaranteed uniform spacing between compounds, so it's one-hot encoded. drop_first=True drops HARD as the reference category to avoid perfect collinearity among the dummies and the intercept in linear models. Each remaining column lets the model learn that compound's pace/degradation effect relative to the HARD baseline.

In [ ]:
finisher_laps = pd.get_dummies(finisher_laps, columns=["Compound"], drop_first=True)
print(finisher_laps.head())
print(finisher_laps.columns)

### 3.4 Nonlinearity feature (TyreLife squared)
Adding a squared tyre-age feature to capture nonlinear degradation. Degradation is nonlinear (gentle early, steeper late), so a quadratic term lets the degradation slope change with tyre age. Squared only TyreLife (the curved variable), not FuelMass (linear). It remains linear regression (linear in coefficients); the squared term is an engineered polynomial feature.

In [ ]:
finisher_laps["TyreLifeSquared"] = finisher_laps["TyreLife"] ** 2
finisher_laps[["TyreLife", "TyreLifeSquared"]]

### 3.5 Weather feature (track temperature)
_(To do: pull session.weather_data, align it to laps by nearest timestamp (e.g. merge_asof on LapStartTime), and add TrackTemp as a feature. Most informative across circuits with different thermal conditions.)_

In [ ]:
weather_data = bahrain2023_race.weather_data
print(weather_data)
print(bahrain2023_race.weather_data.columns)

In [ ]:
laps_sorted = finisher_laps.sort_values("LapStartTime")
weather_sorted = weather_data.sort_values("Time")
merged = pd.merge_asof(laps_sorted, weather_sorted, left_on="LapStartTime", right_on="Time", direction="nearest")
print(merged.isna().sum())

## Phase 4: Generalising across races
All single-race steps above are consolidated into one reusable function, in the same chronological order they were developed: load -> clean -> identify finishers (official Status) -> fuel mass -> estimate gamma -> raw degradation plots -> fuel correction (+ corrected plots) -> weather alignment (TrackTemp) -> compound one-hot -> TyreLifeSquared. It prints the meaningful info (status, missing values, finishers/retirees, per-driver coefficients, gamma) and saves the per-stint plots, returning the model-ready finisher laps and the race gamma.

In [5]:
def race_analysis(gp, year, plot=True):

    race = fastf1.get_session(year, gp, "R")
    race.load()
    laps = race.laps
    results = race.results
    weather_data = race.weather_data

    # ---------- CLEAN ----------
    laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()
    laps["Stint"] = laps["Stint"].astype(int)
    clean_laps = laps[laps["IsAccurate"] == True].copy()

    print(f"=== {gp} {year} ===")
    #print(f"Status values: {results['Status'].unique()}")
    #print(f"Missing values (key cols):\n{clean_laps[['LapTimeSeconds','TyreLife','Compound','Stint','LapNumber']].isna().sum()}")

    # ---------- FINISHERS (official Status: Finished + Lapped) ----------
    finisher_abbrevs = results[results["Status"].isin(["Finished", "Lapped"])]["Abbreviation"].tolist()
    finisher_laps = clean_laps[clean_laps["Driver"].isin(finisher_abbrevs)].copy()
    drivers = finisher_laps["Driver"].unique()
    print(f"All drivers: {clean_laps['Driver'].nunique()}, finishers: {finisher_laps['Driver'].nunique()}")
    print(f"Dropped (retired): {set(clean_laps['Driver'].unique()) - set(finisher_abbrevs)}")

    # ---------- FUEL MASS ----------
    START_FUEL = 109.0
    RACE_LAPS = clean_laps["LapNumber"].max()
    FUEL_BURN = START_FUEL / RACE_LAPS
    #print(f"Race laps: {RACE_LAPS}")
    finisher_laps["FuelMass"] = START_FUEL - (FUEL_BURN * finisher_laps["LapNumber"]) + 1

    # ---------- ESTIMATE GAMMA (finishers only; pit resets decouple fuel from tyre) ----------
    sec_per_kg_fuel = []
    for d in drivers:
        driver_laps = finisher_laps[finisher_laps["Driver"] == d].copy()
        y = driver_laps["LapTimeSeconds"]
        X = driver_laps[["TyreLife", "FuelMass"]]
        laptime_predictor = LinearRegression()
        laptime_predictor.fit(X, y)
        #print(f"Driver: {d}, deg_rate = {laptime_predictor.coef_[0]}, fuel_coef = {laptime_predictor.coef_[1]}")
        sec_per_kg_fuel.append(laptime_predictor.coef_[1])
    gamma = np.mean(sec_per_kg_fuel)
    per_lap_gain = gamma * FUEL_BURN
    race_cum_gain = per_lap_gain * RACE_LAPS
    print(f"Mean of seconds earned per kg (gamma): {gamma}")
    print(f"Per-lap gain in seconds: {per_lap_gain}")
    print(f"Race gain in seconds: {race_cum_gain}")

    # ---------- RAW DEGRADATION PLOTS (per stint, line + scatter) ----------
    if plot:
        for d in drivers:
            ds = finisher_laps[finisher_laps["Driver"] == d].Stint.unique()
            for s in ds:
                driver_stint = finisher_laps[(finisher_laps["Driver"] == d) & (finisher_laps["Stint"] == s)]
                compound = driver_stint["Compound"].iloc[0]
                plt.xlabel("Tyre Age")
                plt.ylabel("Lap Time")
                plt.title(f"{d} Stint {s} Compound: {compound} {gp} {year} laptime evolution by tyre age (Lower is faster)")
                plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
                plt.scatter(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
                plt.savefig(f"../figs/{gp}{year}_{d}_stint{s}_{compound}_tyredeg_evo_scatter.png")
                plt.cla()
        plt.close()

    # ---------- FUEL CORRECTION (+ corrected degradation plots, line + scatter) ----------
    finisher_laps["FuelCorrectedLapTime"] = finisher_laps["LapTimeSeconds"] - gamma * finisher_laps["FuelMass"]
    #print(finisher_laps[["LapNumber", "Driver", "LapTimeSeconds", "FuelMass", "FuelCorrectedLapTime"]])
    if plot:
        for d in drivers:
            ds = finisher_laps[finisher_laps["Driver"] == d].Stint.unique()
            for s in ds:
                driver_stint = finisher_laps[(finisher_laps["Driver"] == d) & (finisher_laps["Stint"] == s)]
                compound = driver_stint["Compound"].iloc[0]
                plt.xlabel("Tyre Age")
                plt.ylabel("Lap Time")
                plt.title(f"{d} Stint {s} Compound: {compound} {gp} {year} laptime evolution by tyre age (Fuel-corrected) (Lower is faster)")
                plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['FuelCorrectedLapTime']))
                plt.scatter(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['FuelCorrectedLapTime']))
                plt.savefig(f"../figs/{gp}{year}_{d}_stint{s}_{compound}_tyredeg_evo_fuel_corrected_scatter.png")
                plt.cla()
        plt.close()

    # ---------- WEATHER (align track temp etc. to laps by nearest timestamp) ----------
    laps_sorted = finisher_laps.sort_values("LapStartTime")
    weather_sorted = weather_data.sort_values("Time")
    finisher_laps = pd.merge_asof(laps_sorted, weather_sorted, left_on="LapStartTime", right_on="Time", direction="nearest")
    #print(f"Missing after weather merge (TrackTemp): {finisher_laps['TrackTemp'].isna().sum()}")

    # ---------- FEATURE ENGINEERING (compound one-hot, TyreLife squared) ----------
    # finisher_laps = pd.get_dummies(finisher_laps, columns=["Compound"], drop_first=True)
    finisher_laps["TyreLifeSquared"] = finisher_laps["TyreLife"] ** 2
    #print(finisher_laps.columns)

    return finisher_laps, gamma


#race_data, race_gamma = race_analysis("Las Vegas Grand Prix", 2024, plot=False)
#print(f"Gamma: {race_gamma}")
#print(f"Race Data:\n{race_data}")
#print(race_data["Rainfall"].mean())


### 4.1 Multi-race screening and dataset creation (with rate-limit handling)


In [ ]:
logging.getLogger('fastf1').setLevel(logging.WARNING)
CALLS_PER_RACE_EST = 6
CALLS_CAP = 490
SLEEP_BUFFER_MIN = 5
GAMMA_MIN, GAMMA_MAX = 0.015, 0.06
MAX_RAIN_FRACTION = 0.10
MIN_FINISHERS = 8
YEARS = [2022, 2023, 2024, 2025]
PROGRESS_FILE = "screen_progress.json"

progress = json.load(open(PROGRESS_FILE)) if os.path.exists(PROGRESS_FILE) else {}
ok_frames = []
calls = 0

for year in YEARS:
    try:
        schedule = fastf1.get_event_schedule(year)
    except Exception:
        continue

    for i, event in schedule.iterrows():
        if event.get("RoundNumber", 0) < 1:
            continue
        gp = event["EventName"]
        k = f"{year}|{gp}"

        if calls + CALLS_PER_RACE_EST > CALLS_CAP:
            now = dt.datetime.now()
            nxt = (now + dt.timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
            time.sleep((nxt - now).total_seconds() + SLEEP_BUFFER_MIN * 60)
            calls = 0

        if k in progress and progress[k]["verdict"] == "DROP":
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                laps_df, gamma = race_analysis(gp, year, plot=False)
            calls += CALLS_PER_RACE_EST
        except Exception as e:
            if "RateLimit" in type(e).__name__:
                now = dt.datetime.now()
                nxt = (now + dt.timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
                time.sleep((nxt - now).total_seconds() + SLEEP_BUFFER_MIN * 60)
                calls = 0
                continue
            progress[k] = {"verdict": "DROP", "reason": f"error: {type(e).__name__}", "n_finishers": None, "rain_fraction": None, "gamma": None, "year": year, "gp": gp}
            continue

        n = laps_df["Driver"].nunique()
        rf = float(laps_df["Rainfall"].mean()) if "Rainfall" in laps_df.columns else 0.0
        verdict, reason = "DROP", ""
        if n < MIN_FINISHERS:
            reason = f"too few finishers ({n})"
        elif rf > MAX_RAIN_FRACTION:
            reason = f"too wet ({rf:.2f})"
        elif not (GAMMA_MIN <= gamma <= GAMMA_MAX):
            reason = f"non-physical gamma ({gamma:.4f})"
        else:
            verdict, reason = "OK", "passed"

        progress[k] = {"verdict": verdict, "reason": reason, "n_finishers": int(n), "rain_fraction": rf, "gamma": float(gamma), "year": year, "gp": gp}

        if verdict == "OK":
            laps_df = laps_df.copy()
            laps_df["Race"] = f"{year}_{gp}"
            ok_frames.append(laps_df)

    json.dump(progress, open(PROGRESS_FILE, "w"), indent=1)

all_laps = pd.concat(ok_frames, ignore_index=True)
team_map = {
    "AlphaTauri": "RB",
    "Racing Bulls": "RB",
    "Alfa Romeo": "Kick Sauber",
}
all_laps["Team"] = all_laps["Team"].replace(team_map)
all_laps = pd.get_dummies(all_laps, columns=["Compound", "Team"], drop_first=True)

print("all_laps shape:", all_laps.shape)
print("\nraces:", all_laps["Race"].nunique())
print(all_laps["Race"].value_counts())
print("\ncolumns:", list(all_laps.columns))

core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.


=== Bahrain Grand Prix 2022 ===
All drivers: 20, finishers: 17
Dropped (retired): {'GAS', 'VER', 'PER'}
Mean of seconds earned per kg (gamma): 0.04318842609029687
Per-lap gain in seconds: 0.08258839375162033
Race gain in seconds: 4.707538443842359


core        WARNING 	No lap data for driver 22
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)


=== Saudi Arabian Grand Prix 2022 ===
All drivers: 18, finishers: 12
Dropped (retired): {'ALO', 'LAT', 'ALB', 'STR', 'BOT', 'RIC'}
Mean of seconds earned per kg (gamma): 0.03683735576459308
Per-lap gain in seconds: 0.08030543556681291
Race gain in seconds: 4.0152717783406455


core        WARNING 	Driver 16 completed the race distance 00:00.140000 before the recorded end of the session.


...

=== Singapore Grand Prix 2025 ===
All drivers: 20, finishers: 20
Dropped (retired): set()
Mean of seconds earned per kg (gamma): 0.02755044520984931
Per-lap gain in seconds: 0.048435460126993145
Race gain in seconds: 3.002998527873575
=== United States Grand Prix 2025 ===
All drivers: 20, finishers: 19
Dropped (retired): {'SAI'}
Mean of seconds earned per kg (gamma): 0.029337267848658
Per-lap gain in seconds: 0.05710289634828075
Race gain in seconds: 3.197762195503722
=== Mexico City Grand Prix 2025 ===
All drivers: 20, finishers: 16
Dropped (retired): {'LAW', 'HUL', 'ALO', 'SAI'}
Mean of seconds earned per kg (gamma): 0.03038046298897832
Per-lap gain in seconds: 0.04664042909575545
Race gain in seconds: 3.311470465798637


core        WARNING 	Driver 4 completed the race distance 00:00.010000 before the recorded end of the session.
core        WARNING 	Fixed incorrect tyre stint information for driver '63'


=== São Paulo Grand Prix 2025 ===
All drivers: 18, finishers: 17
Dropped (retired): {'HAM'}
Mean of seconds earned per kg (gamma): 0.022555357877657046
Per-lap gain in seconds: 0.03462723955865659
Race gain in seconds: 2.458534008664618


events      WARNING 	Correcting user input 'Qatar Grand Prix' to 'Qatar Grand Prix'


=== Las Vegas Grand Prix 2025 ===
All drivers: 18, finishers: 15
Dropped (retired): {'NOR', 'ALB', 'PIA'}
Mean of seconds earned per kg (gamma): 0.03372516341117593
Per-lap gain in seconds: 0.07352085623636354
Race gain in seconds: 3.676042811818177
=== Qatar Grand Prix 2025 ===
All drivers: 20, finishers: 16
Dropped (retired): {'HAD', 'HUL', 'BEA', 'STR'}
Mean of seconds earned per kg (gamma): 0.04347348485122769
Per-lap gain in seconds: 0.08313350611901435
Race gain in seconds: 4.738609848783818
=== Abu Dhabi Grand Prix 2025 ===
All drivers: 20, finishers: 20
Dropped (retired): set()
Mean of seconds earned per kg (gamma): 0.026035158110350315
Per-lap gain in seconds: 0.04892814196600318
Race gain in seconds: 2.8378322340281845
all_laps shape: (61185, 58)

races: 69
Race
2024_Dutch Grand Prix            1354
2024_Austrian Grand Prix         1269
2025_Hungarian Grand Prix        1245
2025_Monaco Grand Prix           1238
2024_Hungarian Grand Prix        1225
                           

## Phase 5: Modelling
Target: lap time (seconds). Features: TyreLife, TyreLifeSquared, FuelMass, Compound (one-hot), Team (one-hot), [TrackTemp]. The tyre-age dependence the model learns is the degradation; the fuel dependence is the fuel effect.

### 5.1 Train/test split (by race)
_(To do: split by race — train on some races, test on a held-out race — to avoid leakage from time-adjacent near-duplicate laps. Define X_train/y_train and X_test/y_test.)_

### 5.2 Baseline
_(To do: predict the mean training lap time for every test lap; report MAE. This is the bar the models must beat.)_

### 5.3 Model 1 — Linear Regression
_(To do: fit on X_train/y_train, predict on X_test, report MAE.)_

### 5.4 Model 2 — Tree-based model
_(To do: fit a Decision Tree / Random Forest on the same split, report MAE.)_

## Phase 6: Evaluation
_(To do: comparison table — baseline vs Linear vs Tree on test MAE; pick final model with justification; error analysis — where does the model fail, e.g. on the unseen circuit / track-dependent degradation.)_

## Phase 7: Pit Decision (MDP framing)
The pit-stop decision is a sequential decision under uncertainty — an MDP (state: tyre age, fuel, compound, conditions; actions: stay / pit; reward: race time / track position).

_(To do — greedy one-step policy: at a given lap, use the degradation model to project 'stay out' vs 'pit now' (pit cost + fresh-tyre pace) over a short horizon, and recommend the lower-time option. This is a tractable approximation to a full MDP solution.)_

### 7.1 (Optional, if time) Full MDP solution
_(Placeholder for later: define states/actions/transition/reward and solve via value/policy iteration. Requires a transition model and rival-reaction assumptions beyond public data — to be attempted only if time allows, as an extension to the greedy policy above.)_

## Limitations
_(To do: linear-ish fuel burn & assumed start fuel; average gamma; corrected times are fuel-normalised, not physical; degradation is additionally driven by track temperature, surface, layout, driver style (confounded with car), and track evolution — not all modelled; era-specific (2022-2025), cross-era requires retraining; greedy policy is not a full MDP solution; single-car projection without rival reaction. Disclose AI assistance per academic integrity.)_